<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 14 · Caps, Floors, and Swaptions

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch14_caps_floors_swaptions.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

Caps, floors, and swaptions turn forwards, strikes, discount factors, accrual periods, and volatility quotes into option values. In this notebook you reproduce the chapter's Black-style caplet valuations from frozen inputs, sum them into a cap value, and price a payer swaption from its forward swap rate and annuity. You learn the caplet decomposition as notional times discount factor times accrual times the Black call, the meaning of annualized Black volatility in decimal form, how a bumped vega cross-checks the analytic Black vega, and why lognormal pricing needs care when forwards are near zero or negative.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script prices four caplets with the Black-76 formula from the frozen input file, prints each caplet value on a 10 million notional, and reports the cap value as the sum of the caplets. It also prices a one-year payer swaption from the frozen forward swap rate, strike, annuity, and volatility.

In [ ]:
SCRIPT = ROOT / "code" / "ch14_caps_floors_swaptions.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Trace every printed number back to the value chain.

- Confirm the structure: each caplet value is notional times discount factor times accrual times the Black call on the forward; the semiannual accrual of 0.5 halves each option payoff.
- Check the units: volatilities of 0.22 to 0.25 are annualized decimals (22% to 25%), and discount factors run from 0.965 to 0.904 across payment dates.
- Verify the decomposition: the cap value of 77,340.05 is exactly the sum of 7,468.60, 14,828.03, 24,275.43, and 30,767.98.
- Read the swaption line: the payer swaption of 158,874.19 scales the Black call on the 4.1% forward swap rate versus the 4.0% strike by the 10 million notional times the 3.62 annuity.
- Note the model domain: the Black function requires positive forwards and strikes, so negative-rate quotes need a normal or shifted-lognormal convention instead.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch14_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: Black caplet premium and its vega

This mini-lab isolates one caplet, the first period of the chapter cap, and computes its Black-76 premium together with the analytic vega, then cross-checks the vega with a one-vol-point bump. The inputs are frozen from the chapter: notional 10 million, discount factor 0.965, semiannual accrual 0.5, forward 3.8%, strike 4.0%, annualized volatility 22%, and half-year expiry.

In [ ]:
from math import erf, exp, log, pi, sqrt
def black(F, K, vol, T):  # Black-76 call and vega
    root = vol * sqrt(T)
    d1 = (log(F / K) + 0.5 * root ** 2) / root
    c = lambda x: 0.5 * (1 + erf(x / sqrt(2)))  # normal CDF
    return (F * c(d1) - K * c(d1 - root),
            F * sqrt(T) * exp(-0.5 * d1 ** 2) / sqrt(2 * pi))
N, D, a = 10_000_000, 0.965, 0.5  # notional, discount, accrual
F, K, vol, T = 0.038, 0.040, 0.22, 0.5  # frozen inputs
s = N * D * a
value, vega = [s * x for x in black(F, K, vol, T)]
bump = s * black(F, K, vol + 0.01, T)[0] - value
print("premium:", round(value, 2), " vega:", round(vega, 2))
print("one vol point:", round(0.01 * vega, 2), " bump:", round(bump, 2))

### What to observe

The caplet premium is 7,468.60. The analytic vega is 50,106.31 per unit of volatility, so one vol point (0.01) is worth about 501.06, and the direct bump from 0.22 to 0.23 gives 502.18; the small difference is volatility convexity, not an error. With the forward below the strike, the premium is pure time value, which is why vega, not intrinsic, drives the price.

## Exercises

1. **Build the floor.** Convert the four caplets into floorlets with the same strikes and compute the floor value. *Hint:* Use the Black put, strike * N(-d2) - forward * N(-d1); the discount factors and accruals stay unchanged.
2. **Extend the cap.** Add a fifth period with forward 0.043, strike 0.040, discount 0.884, accrual 0.5, volatility 0.26, and expiry 2.5 years, and update the cap value. *Hint:* Reuse the Black call and add the new caplet to the existing sum of 77,340.05.
3. **Reprice the swaption as a receiver.** Turn the payer swaption into a receiver swaption on the same inputs. *Hint:* Replace the call with a put on the forward swap rate; with the swap rate above the strike, the receiver value is much smaller.
4. **Break the volatility units.** Recompute caplet 1 with volatility quoted as 22 basis points (0.0022) instead of 22% and report the damage. *Hint:* The premium collapses toward the out-of-the-money intrinsic value of zero; this is the unit trap the chapter warns about.
5. **Explain the maturity profile.** Explain why caplet 4 is worth roughly four times caplet 1. *Hint:* Total volatility, vol * sqrt(expiry), more than doubles, and the forwards of periods 3 and 4 sit above the strike, which adds intrinsic content.

For more detail, see Chapter 14 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">